# 02 — Data Quality Assessment

## Objective

Assess the quality, consistency, completeness, and integrity of the Synthea healthcare dataset before performing analysis or machine learning.

### Quality dimensions

1. Completeness — Are important fields missing?
2. Uniqueness — Are records duplicated?
3. Validity — Are values within acceptable ranges?
4. Consistency — Do related fields agree with each other?
5. Integrity — Do foreign keys correctly reference other tables?
6. Temporal validity — Are dates logical?
7. Data types — Are columns stored in appropriate formats?

The results of this notebook will guide data cleaning, SQL transformations, feature engineering, and downstream modeling.
Each cell should answer one specific data-quality question.

In [1]:
from pathlib import Path
import pandas as pd
import numpy as np

In [2]:
# Define our data location

data_path = Path("../data/raw/synthea")

csv_files = sorted(data_path.glob("*.csv"))

print(f"Data directory: {data_path.resolve()}")
print(f"Number of CSV tables: {len(csv_files)}")

Data directory: C:\Users\USER\Documents\DATA SCIENCE\healthcare-risk-system\data\raw\synthea
Number of CSV tables: 18


## Load the tables

In [3]:
tables = {}

for file in csv_files:
    tables[file.stem] = pd.read_csv(file)

print(f"Loaded {len(tables)} tables.")

Loaded 18 tables.


In [4]:
# Confirm the loaded dataframe

for name, df in tables.items():
    print(f"{name:25} {df.shape}")

allergies                 (5375, 15)
careplans                 (18976, 9)
claims                    (629678, 31)
claims_transactions       (5547678, 33)
conditions                (208363, 7)
devices                   (33119, 7)
encounters                (337227, 15)
imaging_studies           (481554, 13)
immunizations             (82882, 6)
medications               (292451, 13)
observations              (4355508, 9)
organizations             (1090, 12)
patients                  (5755, 28)
payer_transitions         (213917, 8)
payers                    (10, 22)
procedures                (932402, 10)
providers                 (1090, 14)
supplies                  (151616, 6)


## 1. Completeness — Missing Values

Missing values can affect analysis, joins, statistical calculations, and machine-learning models.

We will calculate:

- Number of missing values
- Percentage of missing values
- Number of columns containing missing values

The goal is to identify which fields require attention before downstream analysis.

In [5]:
missing_reports = {}

for name, df in tables.items():
    missing = df.isna().sum()
    missing_pct = (missing / len(df)) * 100

    report = pd.DataFrame({
        "missing_count": missing,
        "missing_pct": missing_pct
    })

    report = report[report["missing_count"] > 0]
    report = report.sort_values("missing_pct", ascending=False)

    missing_reports[name] = report

print("Missing-value audit completed.")

Missing-value audit completed.


In [6]:
for name, report in missing_reports.items():
    print(f"\n{'=' * 60}")
    print(f"TABLE: {name}")
    print(f"{'=' * 60}")
    print(report)


TABLE: allergies
              missing_count  missing_pct
STOP                   5375   100.000000
REACTION2              3955    73.581395
DESCRIPTION2           3955    73.581395
SEVERITY2              3955    73.581395
REACTION1              3029    56.353488
DESCRIPTION1           3029    56.353488
SEVERITY1              3029    56.353488

TABLE: careplans
                   missing_count  missing_pct
STOP                        9368    49.367622
REASONCODE                  9017    47.517917
REASONDESCRIPTION           9017    47.517917

TABLE: claims
                             missing_count  missing_pct
REFERRINGPROVIDERID                 629678   100.000000
DIAGNOSIS8                          629538    99.977766
DIAGNOSIS7                          628821    99.863899
DIAGNOSIS6                          627432    99.643310
DIAGNOSIS5                          623817    99.069207
DIAGNOSIS4                          610701    96.986237
DIAGNOSIS3                          566555   

## Missingness Summary Across Tables

We now aggregate the missing-value audit into a table so that we can compare data-quality patterns across all 18 datasets.

In [7]:
missing_summary = []

for name, df in tables.items():
    total_cells = df.shape[0] * df.shape[1]
    missing_cells = df.isna().sum().sum()

    missing_summary.append({
        "table": name,
        "rows": df.shape[0],
        "columns": df.shape[1],
        "missing_cells": missing_cells,
        "missing_pct": (missing_cells / total_cells) * 100,
        "columns_with_missing": df.isna().any().sum()
    })

missing_summary_df = (
    pd.DataFrame(missing_summary)
    .sort_values("missing_pct", ascending=False)
    .reset_index(drop=True)
)

missing_summary_df

,table,rows,columns,missing_cells,missing_pct,columns_with_missing
0,allergies,5375,15,26327,32.653643,7
1,claims,629678,31,5855387,29.996832,13
2,claims_transactions,5547678,33,48757734,26.632892,12
3,payers,10,22,50,22.727273,5
4,payer_transitions,213917,8,355871,20.794923,4
5,careplans,18976,9,27402,16.044829,3
6,patients,5755,28,22612,14.032518,9
7,procedures,932402,10,995946,10.681509,2
8,devices,33119,7,12032,5.189943,1
9,encounters,337227,15,251124,4.964490,2


Missingness has been quantified across all 18 Synthea tables. The highest overall missing-cell percentages occur in allergies, claims, claims_transactions, payers, and payer_transitions. However, missingness alone does not establish a data-quality problem; the next step is to examine missingness at the column level and determine whether the missing values are structurally expected or require treatment.

## 2. Uniqueness — Duplicate Records

Duplicate records can cause double-counting, distorted statistics, incorrect joins, and biased machine-learning features.

Distinguish between:

- Exact duplicate rows: every column has the same value.
- Duplicate identifiers: a record's expected unique ID appears more than once.
- Repeated foreign keys: expected in event tables and therefore not automatically considered duplicates.

In [8]:
duplicate_summary = []

for name, df in tables.items():
    exact_duplicates = df.duplicated().sum()

    duplicate_summary.append({
        "table": name,
        "rows": len(df),
        "exact_duplicate_rows": exact_duplicates,
        "duplicate_pct": (exact_duplicates / len(df)) * 100
    })

duplicate_summary_df = (
    pd.DataFrame(duplicate_summary)
    .sort_values("exact_duplicate_rows", ascending=False)
    .reset_index(drop=True)
)

duplicate_summary_df

,table,rows,exact_duplicate_rows,duplicate_pct
0,observations,4355508,1085,0.024911
1,supplies,151616,598,0.394417
2,careplans,18976,0,0.000000
3,providers,1090,0,0.000000
4,procedures,932402,0,0.000000
5,payers,10,0,0.000000
6,payer_transitions,213917,0,0.000000
7,patients,5755,0,0.000000
8,organizations,1090,0,0.000000
9,allergies,5375,0,0.000000


Exact duplicate rows are extremely rare in the Synthea dataset, with duplicates detected only in observations and supplies.

Exact duplicates ≠ duplicate identifiers.

For example, suppose observations contains:

DATE        PATIENT       DESCRIPTION      VALUE

2020-01-01  patient-001   Blood Pressure   120

2020-01-01  patient-001   Blood Pressure   120

Those two rows are exact duplicates.

But this:

DATE        PATIENT       DESCRIPTION      VALUE

2020-01-01  patient-001   Blood Pressure   120

2020-01-02  patient-001   Blood Pressure   118

is not an exact duplicate — even though the same patient has multiple observations.

In [9]:
## Look only at tables that actually have duplicates

duplicate_summary_df[
    duplicate_summary_df["exact_duplicate_rows"] > 0
]

,table,rows,exact_duplicate_rows,duplicate_pct
0,observations,4355508,1085,0.024911
1,supplies,151616,598,0.394417


In [10]:
# Investigate identifiers, identify all tables containing Id:

id_columns = {}

for name, df in tables.items():
    if "Id" in df.columns:
        id_columns[name] = "Id"

id_columns

{'careplans': 'Id',
 'claims': 'Id',
 'encounters': 'Id',
 'imaging_studies': 'Id',
 'organizations': 'Id',
 'patients': 'Id',
 'payers': 'Id',
 'providers': 'Id'}

In [11]:
# check whether those IDs are actually unique.

id_duplicate_summary = []

for name, id_col in id_columns.items():
    df = tables[name]

    duplicate_ids = df[id_col].duplicated().sum()
    null_ids = df[id_col].isna().sum()

    id_duplicate_summary.append({
        "table": name,
        "id_column": id_col,
        "rows": len(df),
        "unique_ids": df[id_col].nunique(),
        "duplicate_ids": duplicate_ids,
        "null_ids": null_ids
    })

id_duplicate_summary_df = pd.DataFrame(id_duplicate_summary)

id_duplicate_summary_df

,table,id_column,rows,unique_ids,duplicate_ids,null_ids
0,careplans,Id,18976,18976,0,0
1,claims,Id,629678,629678,0,0
2,encounters,Id,337227,337227,0,0
3,imaging_studies,Id,481554,27240,454314,0
4,organizations,Id,1090,1090,0,0
5,patients,Id,5755,5755,0,0
6,payers,Id,10,10,0,0
7,providers,Id,1090,1090,0,0


The right action is to investigate why Id repeats before deciding whether they are legitimate or a data-quality issue.

Your result is striking:

Rows: 481,554

Unique Id: 27,240

Duplicate IDs: 454,314

Null IDs: 0

That means the Id column is not behaving like a unique row identifier in this table.

In [12]:
# confirm the duplicates

imaging = tables["imaging_studies"]

imaging["Id"].duplicated().sum()

np.int64(454314)

In [13]:
# Then inspect one duplicated ID:

duplicate_id = imaging.loc[
    imaging["Id"].duplicated(), "Id"
].iloc[0]

imaging[imaging["Id"] == duplicate_id]

,Id,DATE,PATIENT,ENCOUNTER,SERIES_UID,BODYSITE_CODE,BODYSITE_DESCRIPTION,MODALITY_CODE,MODALITY_DESCRIPTION,INSTANCE_UID,SOP_CODE,SOP_DESCRIPTION,PROCEDURE_CODE
25,b03ba64b-f68e-5df3-e3be-587b3e90b0ce,2022-07-06T08:47:00Z,b03ba64b-f68e-5df3-072b-c6665b8117c9,b03ba64b-f68e-5df3-f177-37e499a724ba,1.2.840.99999999.1.52757043.1657097220370,181171005,Entire retina (body structure),OP,Ophthalmic Photography,1.2.840.99999999.1.1.61941673.1657097220370,1.2.840.10008.5.1.4.1.1.77.1.5.1,Ophthalmic Photography 8 Bit Image Storage,314971001
26,b03ba64b-f68e-5df3-e3be-587b3e90b0ce,2022-07-06T08:47:00Z,b03ba64b-f68e-5df3-072b-c6665b8117c9,b03ba64b-f68e-5df3-f177-37e499a724ba,1.2.840.99999999.1.52757043.1657097220370,181171005,Entire retina (body structure),OP,Ophthalmic Photography,1.2.840.99999999.1.2.85250543.1657097220370,1.2.840.10008.5.1.4.1.1.77.1.5.1,Ophthalmic Photography 8 Bit Image Storage,314971001


In [14]:
# Check whether the rows are actually identical

imaging[imaging["Id"] == duplicate_id].duplicated().sum()

np.int64(0)

The repeated Id values are not caused by identical duplicate records.

Instead, the same Id is associated with multiple distinct records.

So do not delete the 454,314 rows as that would potentially destroy legitimate imaging data.

Finding: imaging_studies.Id contains a large number of repeated values. Investigation of sampled duplicate IDs shows that the corresponding rows are not exact duplicates; they contain distinct imaging-instance information. Therefore, repeated Id values should not be treated as duplicate records or removed. The Id field should not be assumed to be a unique row-level identifier for imaging_studies. The table's identifier structure should be examined before establishing primary-key constraints.

In [15]:
# find out what actually makes an imaging row unique

imaging.columns.tolist()

['Id',
 'DATE',
 'PATIENT',
 'ENCOUNTER',
 'SERIES_UID',
 'BODYSITE_CODE',
 'BODYSITE_DESCRIPTION',
 'MODALITY_CODE',
 'MODALITY_DESCRIPTION',
 'INSTANCE_UID',
 'SOP_CODE',
 'SOP_DESCRIPTION',
 'PROCEDURE_CODE']

In [16]:
# This is especially useful because it will display those two records vertically, making it much easier to see exactly which columns distinguish them.

imaging[imaging["Id"] == duplicate_id].T

,25,26
Id,b03ba64b-f68e-5df3-e3be-587b3e90b0ce,b03ba64b-f68e-5df3-e3be-587b3e90b0ce
DATE,2022-07-06T08:47:00Z,2022-07-06T08:47:00Z
PATIENT,b03ba64b-f68e-5df3-072b-c6665b8117c9,b03ba64b-f68e-5df3-072b-c6665b8117c9
ENCOUNTER,b03ba64b-f68e-5df3-f177-37e499a724ba,b03ba64b-f68e-5df3-f177-37e499a724ba
SERIES_UID,1.2.840.99999999.1.52757043.1657097220370,1.2.840.99999999.1.52757043.1657097220370
BODYSITE_CODE,181171005,181171005
BODYSITE_DESCRIPTION,Entire retina (body structure),Entire retina (body structure)
MODALITY_CODE,OP,OP
MODALITY_DESCRIPTION,Ophthalmic Photography,Ophthalmic Photography
INSTANCE_UID,1.2.840.99999999.1.1.61941673.1657097220370,1.2.840.99999999.1.2.85250543.1657097220370


Identifier uniqueness finding — imaging_studies: The Id column is not unique across rows. Investigation of repeated IDs shows that records can share the same study-level attributes while having different INSTANCE_UID values. Therefore, repeated Id values represent distinct imaging instances rather than duplicate records. Id should not be used as the row-level primary key, and records should not be removed solely because Id is repeated.

In [17]:
# test whether INSTANCE_UID is actually unique.

imaging["INSTANCE_UID"].nunique()

481554

In [18]:
imaging["INSTANCE_UID"].duplicated().sum()

np.int64(0)

imaging_studies.Id is non-unique and therefore unsuitable as the row-level primary key. INSTANCE_UID is unique across all 481,554 records and contains no null values, making it an appropriate candidate for the row-level identifier.

In [19]:
# Investigate the duplicate rows

observations = tables["observations"]

duplicate_observations = observations[
    observations.duplicated(keep=False)
]

duplicate_observations.head(20)

,DATE,PATIENT,ENCOUNTER,CATEGORY,CODE,DESCRIPTION,VALUE,UNITS,TYPE
40548,2018-11-28T12:20:29Z,7f8f9677-e4aa-d47a-003b-fa864f59b70d,7f8f9677-e4aa-d47a-29b0-8bfaf6d13be3,laboratory,2339-0,Glucose [Mass/volume] in Blood,77.4,mg/dL,numeric
40549,2018-11-28T12:20:29Z,7f8f9677-e4aa-d47a-003b-fa864f59b70d,7f8f9677-e4aa-d47a-29b0-8bfaf6d13be3,laboratory,6299-2,Urea nitrogen [Mass/volume] in Blood,16.3,mg/dL,numeric
40551,2018-11-28T12:20:29Z,7f8f9677-e4aa-d47a-003b-fa864f59b70d,7f8f9677-e4aa-d47a-29b0-8bfaf6d13be3,laboratory,49765-1,Calcium [Mass/volume] in Blood,9.1,mg/dL,numeric
40552,2018-11-28T12:20:29Z,7f8f9677-e4aa-d47a-003b-fa864f59b70d,7f8f9677-e4aa-d47a-29b0-8bfaf6d13be3,laboratory,2947-0,Sodium [Moles/volume] in Blood,141.5,mmol/L,numeric
40553,2018-11-28T12:20:29Z,7f8f9677-e4aa-d47a-003b-fa864f59b70d,7f8f9677-e4aa-d47a-29b0-8bfaf6d13be3,laboratory,6298-4,Potassium [Moles/volume] in Blood,4.5,mmol/L,numeric
40554,2018-11-28T12:20:29Z,7f8f9677-e4aa-d47a-003b-fa864f59b70d,7f8f9677-e4aa-d47a-29b0-8bfaf6d13be3,laboratory,2069-3,Chloride [Moles/volume] in Blood,109.3,mmol/L,numeric
40555,2018-11-28T12:20:29Z,7f8f9677-e4aa-d47a-003b-fa864f59b70d,7f8f9677-e4aa-d47a-29b0-8bfaf6d13be3,laboratory,20565-8,Carbon dioxide total [Moles/volume] in Blood,20.4,mmol/L,numeric
40572,2018-11-28T12:20:29Z,7f8f9677-e4aa-d47a-003b-fa864f59b70d,7f8f9677-e4aa-d47a-29b0-8bfaf6d13be3,laboratory,2339-0,Glucose [Mass/volume] in Blood,77.4,mg/dL,numeric
40573,2018-11-28T12:20:29Z,7f8f9677-e4aa-d47a-003b-fa864f59b70d,7f8f9677-e4aa-d47a-29b0-8bfaf6d13be3,laboratory,6299-2,Urea nitrogen [Mass/volume] in Blood,16.3,mg/dL,numeric
40575,2018-11-28T12:20:29Z,7f8f9677-e4aa-d47a-003b-fa864f59b70d,7f8f9677-e4aa-d47a-29b0-8bfaf6d13be3,laboratory,49765-1,Calcium [Mass/volume] in Blood,9.1,mg/dL,numeric


In [20]:
duplicate_observations.shape

(2170, 9)

In [21]:
# Count how many times each exact observation row appears, then summarize how many rows occur 2, 3, 4, etc. times.
duplicate_counts = (
    observations.value_counts()
    .reset_index(name="count")
)

duplicate_counts["count"].value_counts().sort_index()

count
1    3010511
2       1085
Name: count, dtype: int64

### Exact Duplicate Investigation — Observations

The `observations` table contains **2,170 rows belonging to exact duplicate groups**, representing **1,085 duplicate records beyond their first occurrences**.

Inspection of the duplicated records confirmed that the repeated laboratory observations have identical values across all 9 columns, including the date, patient, encounter, test code, description, value, unit, and type.

Therefore, these records are classified as **true exact duplicates**.

Unlike the `imaging_studies` table, where repeated `Id` values represented distinct records with different `INSTANCE_UID` values, the duplicates identified in `observations` contain no differences between the repeated rows.

**Action:** The raw Synthea data will remain unchanged. These exact duplicates will be considered for removal when creating the cleaned/processed dataset.


## Extract the duplicated supply records

In [22]:
# Extract every supply record that belongs to an exact duplicate group so we can inspect the duplicated records.
supplies = tables["supplies"]

duplicate_supplies = supplies[
    supplies.duplicated(keep=False) # all copies of a duplicated row are marked True.
]

duplicate_supplies.head(20)

,DATE,PATIENT,ENCOUNTER,CODE,DESCRIPTION,QUANTITY
73,2014-12-23,dd590ad1-2bd7-cbcc-e803-d99a0cdbc281,dd590ad1-2bd7-cbcc-5af7-4fde25627035,337388004,Blood glucose testing strips (physical object),50
74,2014-12-23,dd590ad1-2bd7-cbcc-e803-d99a0cdbc281,dd590ad1-2bd7-cbcc-5af7-4fde25627035,337388004,Blood glucose testing strips (physical object),50
180,2020-11-22,88b855e0-ade8-77b5-2559-0982d3b56b82,88b855e0-ade8-77b5-d172-8676813fd508,337388004,Blood glucose testing strips (physical object),50
181,2020-11-22,88b855e0-ade8-77b5-2559-0982d3b56b82,88b855e0-ade8-77b5-d172-8676813fd508,337388004,Blood glucose testing strips (physical object),50
246,2010-06-06,adf67d43-fafd-499e-f5cf-f88e81fa7562,adf67d43-fafd-499e-725d-db11c8664814,337388004,Blood glucose testing strips (physical object),50
247,2010-06-06,adf67d43-fafd-499e-f5cf-f88e81fa7562,adf67d43-fafd-499e-725d-db11c8664814,337388004,Blood glucose testing strips (physical object),50
350,1990-12-09,60597902-0831-b5fa-f8e0-709d779acf32,60597902-0831-b5fa-c684-06b2de316de9,337388004,Blood glucose testing strips (physical object),50
351,1990-12-09,60597902-0831-b5fa-f8e0-709d779acf32,60597902-0831-b5fa-c684-06b2de316de9,337388004,Blood glucose testing strips (physical object),50
366,2019-11-24,60597902-0831-b5fa-f8e0-709d779acf32,60597902-0831-b5fa-de69-a718818d2c1d,337388004,Blood glucose testing strips (physical object),50
367,2019-11-24,60597902-0831-b5fa-f8e0-709d779acf32,60597902-0831-b5fa-de69-a718818d2c1d,337388004,Blood glucose testing strips (physical object),50


In [23]:
# Count how many times each exact supply record appears, then summarize the sizes of the duplicate groups.
duplicate_supply_counts = (
    supplies.value_counts()
    .reset_index(name="count")
)

duplicate_supply_counts["count"].value_counts().sort_index()

count
1    150439
2       560
3        19
Name: count, dtype: int64

### Exact Duplicate Finding — Supplies

The `supplies` table contains **598 redundant exact-duplicate records**. Investigation of duplicate frequency shows that **560 unique supply records appear twice**, while **19 unique supply records appear three times**.

This means **579 distinct supply records belong to duplicate groups**, involving **1,177 rows in total**. After retaining one copy of each unique record, **598 redundant copies** can be removed.

Inspection of the duplicated records confirms that the repeated rows contain identical values across all columns, indicating that these are **true exact duplicates rather than legitimate repeated supply events**.

**Action:** Preserve the raw Synthea data unchanged and remove the **598 redundant exact-duplicate copies** when creating the processed/cleaned dataset.

Duplicate group = the same record appears more than once.

Duplicate rows = all the rows belonging to that group.

Redundant copies = the extra copies beyond the one we decide to keep.

In [24]:
# List the columns in each table that look like potential identifiers so we can investigate their uniqueness.

for table_name, df in tables.items():
    id_columns = [
        column for column in df.columns
        if column.lower() == "id" or column.lower().endswith("_id")
    ]
    
    print(f"{table_name}: {id_columns}")

allergies: []
careplans: ['Id']
claims: ['Id']
claims_transactions: ['ID']
conditions: []
devices: []
encounters: ['Id']
imaging_studies: ['Id']
immunizations: []
medications: []
observations: []
organizations: ['Id']
patients: ['Id']
payer_transitions: []
payers: ['Id']
procedures: []
providers: ['Id']
supplies: []


### Test whether the candidate Id columns are actually unique
For each table that has an Id/ID column, we want three things:

Total rows

Number of unique IDs

Number of duplicated IDs

Number of missing/null IDs

In [25]:
# Check whether each candidate ID column is unique, duplicated, or missing across its table.
candidate_id_tables = [
    "careplans",
    "claims",
    "claims_transactions",
    "encounters",
    "imaging_studies",
    "organizations",
    "patients",
    "payers",
    "providers"
]

for table_name in candidate_id_tables:
    df = tables[table_name]
    
    # Find the actual ID column, allowing for "Id" and "ID".
    id_column = next(
        column for column in df.columns
        if column.lower() == "id"
    )
    
    total_rows = len(df)
    unique_ids = df[id_column].nunique()
    duplicate_ids = df[id_column].duplicated().sum()
    null_ids = df[id_column].isna().sum()
    
    print(f"\n{table_name}")
    print(f"ID column: {id_column}")
    print(f"Rows: {total_rows:,}")
    print(f"Unique IDs: {unique_ids:,}")
    print(f"Duplicate IDs: {duplicate_ids:,}")
    print(f"Null IDs: {null_ids:,}")


careplans
ID column: Id
Rows: 18,976
Unique IDs: 18,976
Duplicate IDs: 0
Null IDs: 0

claims
ID column: Id
Rows: 629,678
Unique IDs: 629,678
Duplicate IDs: 0
Null IDs: 0

claims_transactions
ID column: ID
Rows: 5,547,678
Unique IDs: 5,547,678
Duplicate IDs: 0
Null IDs: 0

encounters
ID column: Id
Rows: 337,227
Unique IDs: 337,227
Duplicate IDs: 0
Null IDs: 0

imaging_studies
ID column: Id
Rows: 481,554
Unique IDs: 27,240
Duplicate IDs: 454,314
Null IDs: 0

organizations
ID column: Id
Rows: 1,090
Unique IDs: 1,090
Duplicate IDs: 0
Null IDs: 0

patients
ID column: Id
Rows: 5,755
Unique IDs: 5,755
Duplicate IDs: 0
Null IDs: 0

payers
ID column: Id
Rows: 10
Unique IDs: 10
Duplicate IDs: 0
Null IDs: 0

providers
ID column: Id
Rows: 1,090
Unique IDs: 1,090
Duplicate IDs: 0
Null IDs: 0


The important takeaway is:

Most tables with explicit ID columns have valid row-level unique identifiers. imaging_studies.Id is an exception because it represents a higher-level imaging study rather than an individual row. The remaining tables without explicit ID columns require natural/composite-key analysis.

Duplicate identifier ≠ duplicate record.

# Understanding and designing a relational healthcare data model

There are 9 tables without an obvious Id:

allergies
conditions
devices
immunizations
medications
observations
payer_transitions
procedures
supplies

But first,understand the columns in each table. A candidate key only makes sense when we understand what each column represents.

In [26]:
# Display the columns for each table without an explicit ID column so we can identify possible natural or composite identifiers.
tables_without_id = [
    "allergies",
    "conditions",
    "devices",
    "immunizations",
    "medications",
    "observations",
    "payer_transitions",
    "procedures",
    "supplies"
]

for table_name in tables_without_id:
    print(f"\n{'=' * 60}")
    print(f"TABLE: {table_name}")
    print(f"{'=' * 60}")
    print(tables[table_name].columns.tolist())


TABLE: allergies
['START', 'STOP', 'PATIENT', 'ENCOUNTER', 'CODE', 'SYSTEM', 'DESCRIPTION', 'TYPE', 'CATEGORY', 'REACTION1', 'DESCRIPTION1', 'SEVERITY1', 'REACTION2', 'DESCRIPTION2', 'SEVERITY2']

TABLE: conditions
['START', 'STOP', 'PATIENT', 'ENCOUNTER', 'SYSTEM', 'CODE', 'DESCRIPTION']

TABLE: devices
['START', 'STOP', 'PATIENT', 'ENCOUNTER', 'CODE', 'DESCRIPTION', 'UDI']

TABLE: immunizations
['DATE', 'PATIENT', 'ENCOUNTER', 'CODE', 'DESCRIPTION', 'BASE_COST']

TABLE: medications
['START', 'STOP', 'PATIENT', 'PAYER', 'ENCOUNTER', 'CODE', 'DESCRIPTION', 'BASE_COST', 'PAYER_COVERAGE', 'DISPENSES', 'TOTALCOST', 'REASONCODE', 'REASONDESCRIPTION']

TABLE: observations
['DATE', 'PATIENT', 'ENCOUNTER', 'CATEGORY', 'CODE', 'DESCRIPTION', 'VALUE', 'UNITS', 'TYPE']

TABLE: payer_transitions
['PATIENT', 'MEMBERID', 'START_DATE', 'END_DATE', 'PAYER', 'SECONDARY_PAYER', 'PLAN_OWNERSHIP', 'OWNER_NAME']

TABLE: procedures
['START', 'STOP', 'PATIENT', 'ENCOUNTER', 'SYSTEM', 'CODE', 'DESCRIPTION',

In [27]:
# Check whether individual columns in the allergies table are unique enough to serve as possible identifiers.
allergies = tables["allergies"]

for column in ["START", "PATIENT", "ENCOUNTER", "CODE"]:
    print(
        f"{column}: "
        f"unique={allergies[column].nunique():,}, "
        f"duplicates={allergies[column].duplicated().sum():,}, "
        f"nulls={allergies[column].isna().sum():,}"
    )

START: unique=973, duplicates=4,402, nulls=0
PATIENT: unique=991, duplicates=4,384, nulls=0
ENCOUNTER: unique=991, duplicates=4,384, nulls=0
CODE: unique=22, duplicates=5,353, nulls=0


In [28]:
# Test whether PATIENT + START + CODE uniquely identifies each allergy record.
allergy_key = ["PATIENT", "START", "CODE"]

print(f"Total rows: {len(allergies):,}")
print(f"Unique combinations: {allergies[allergy_key].drop_duplicates().shape[0]:,}")
print(
    f"Duplicate combinations: "
    f"{allergies.duplicated(subset=allergy_key).sum():,}"
)

Total rows: 5,375
Unique combinations: 5,375
Duplicate combinations: 0


### `allergies` Identifier Finding

The `allergies` table does not contain an explicit row-level ID column. Individual columns such as `START`, `PATIENT`, `ENCOUNTER`, and `CODE` were tested and none were unique.

A composite key consisting of `PATIENT + START + CODE` was then tested. The combination produced 5,375 unique combinations across 5,375 rows, with zero duplicate combinations and no null values in the component columns.

Therefore, `PATIENT + START + CODE` is a valid candidate composite identifier for the `allergies` table in this Synthea dataset.

No records require removal based on identifier uniqueness.


## Testing composite key for `conditions` table

Now we'll repeat the same reasoning.

conditions has:

START

STOP

PATIENT

ENCOUNTER

SYSTEM

CODE

DESCRIPTION

Let's first test the obvious individual candidates:

In [29]:
# Check whether individual columns in the conditions table are unique enough to identify a condition record.
conditions = tables["conditions"]

for column in ["START", "STOP", "PATIENT", "ENCOUNTER", "CODE"]:
    print(
        f"{column}: "
        f"unique={conditions[column].nunique():,}, "
        f"duplicates={conditions[column].duplicated().sum():,}, "
        f"nulls={conditions[column].isna().sum():,}"
    )

START: unique=19,585, duplicates=188,778, nulls=0
STOP: unique=13,264, duplicates=195,098, nulls=53,792
PATIENT: unique=5,755, duplicates=202,608, nulls=0
ENCOUNTER: unique=133,278, duplicates=75,085, nulls=0
CODE: unique=302, duplicates=208,061, nulls=0


In [30]:
# Test whether PATIENT + START + CODE uniquely identifies each condition record.
condition_key = ["PATIENT", "START", "CODE"]

print(f"Total rows: {len(conditions):,}")
print(
    f"Unique combinations: "
    f"{conditions[condition_key].drop_duplicates().shape[0]:,}"
)
print(
    f"Duplicate combinations: "
    f"{conditions.duplicated(subset=condition_key).sum():,}"
)

Total rows: 208,363
Unique combinations: 208,363
Duplicate combinations: 0


### `conditions` Identifier Finding

The `conditions` table does not contain an explicit row-level ID column. Individual fields such as `START`, `PATIENT`, `ENCOUNTER`, and `CODE` were not unique on their own.

The composite combination `PATIENT + START + CODE` was tested and produced 208,363 unique combinations across 208,363 rows, with zero duplicate combinations.

Therefore, `PATIENT + START + CODE` is a valid candidate composite identifier for the `conditions` table in this Synthea dataset.

The `STOP` column was not required to establish uniqueness and contains 53,792 null values. No records require removal based on identifier uniqueness.
